# Sistemas de Información Ambiental (SIAC)

**Bloque A — Gestión institucional** | Línea: `sistemas_informacion_ambiental`

---

## Contexto institucional

El **Sistema de Información Ambiental de Colombia (SIAC)** es la arquitectura federada de actores, políticas, procesos y tecnologías responsables de gestionar la información ambiental a nivel nacional. Fue creado por la **Ley 99 de 1993** como parte del Sistema Nacional Ambiental (SINA) y es coordinado técnicamente por el IDEAM.

El SIAC no es un portal único: es un ecosistema de subsistemas especializados:

| Subsistema | Qué gestiona |
|---|---|
| **SIRH** | Recurso hídrico (caudales, calidad, usos) |
| **SMByC** | Monitoreo de bosques y carbono (deforestación) |
| **SIB Colombia** | Biodiversidad (especies, ecosistemas) |
| **RUA / RETC** | Emisiones industriales y transferencia de contaminantes |
| **RUNAP** | Áreas protegidas del SPNN |
| **VITAL** | Trámites ambientales en línea |

**¿Por qué analizar estadísticamente el SIAC?**  
La cobertura, densidad y calidad de las estaciones de monitoreo que alimentan estos subsistemas determina directamente la calidad de TODAS las decisiones ambientales que toman las CARs, el MADS y el IDEAM. Una CAR que no puede medir, no puede gestionar.

---

## Preguntas analíticas que responde este notebook

1. ¿Cómo ha evolucionado la cobertura de estaciones de monitoreo activas en Colombia?
2. ¿Sigue un patrón de adopción tecnológica (curva S logística)?
3. ¿Cuál es la proyección de cobertura para los próximos 5 años?
4. ¿Hay diferencias significativas de densidad entre regiones (zona andina vs. Amazonía/Orinoquía)?

---

## Marco normativo relevante

- **Ley 99/1993:** Crea el SINA y el deber de información ambiental.
- **Decreto 1076/2015:** Decreto Único Reglamentario — coordina el SIAC y el SIUR.
- **Resolución 839/2023:** Integra el RETC al RUA operativamente.
- **Ley 1712/2014:** Transparencia y acceso público a información.

> **Documentación de dominio completa:** [`docs/fuentes/sistemas_informacion_ambiental.md`](../../docs/fuentes/sistemas_informacion_ambiental.md)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "sistemas_informacion_ambiental"
VARIABLE = "n_registros"
UNIDAD = ""

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `sistemas_informacion_ambiental` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "sistemas_informacion_ambiental.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### Variable principal: cobertura de estaciones de monitoreo activas (%)

**¿Qué medir?** La variable recomendada para esta línea es la **cobertura de estaciones activas**, expresada como porcentaje del territorio con al menos una estación de monitoreo dentro de un radio de 50 km, calculada anualmente.

Variables derivadas útiles:
- `cobertura_pct`: % del territorio nacional cubierto (0–100%)
- `densidad_est_km2`: número de estaciones por cada 1.000 km²
- `tiempo_al_dato_dias`: mediana de días desde la medición hasta la disponibilidad pública

**Frecuencia natural de la serie:** Anual (los inventarios del IDEAM son anuales). Esto tiene implicaciones metodológicas importantes — ver Sección 7.

**Fuentes de datos reales:**
- Portal IDEAM: http://www.ideam.gov.co/web/oceffa/red-hidrometeorologica
- DHIME (SIRH): http://dhime.ideam.gov.co
- SIAC: http://www.siac.gov.co — módulo de estaciones

> Colocar el archivo en `data/raw/sistemas_informacion_ambiental.csv` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/sistemas_informacion_ambiental.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/sistemas_informacion_ambiental.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "n_registros": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA

### ¿Qué validar en esta línea?

La función `validate()` aplica rangos físicos específicos definidos en `config.py`. Para la cobertura de estaciones, los rangos razonables son:

- `cobertura_pct`: [0, 100] — es un porcentaje, no puede exceder 100%
- `densidad_est_km2`: [0, 10] — Colombia tiene ~1.14 M km², concentración máxima plausible ~5/1000 km²
- `tiempo_al_dato_dias`: [0, 365] — el dato no debería tardar más de un año

**Señales de alerta en los datos del SIAC:**
- Saltos abruptos en cobertura (>15% en un año): probablemente un cambio de metodología de cálculo, no un fenómeno real.
- Datos faltantes: frecuentes en períodos de transición institucional (p. ej., 2020 por pandemia).
- Cobertura decreciente: puede indicar estaciones activas pero no reportando (diferencia entre "instalada" y "activa").

> `validate()` usa rangos físicos específicos para `sistemas_informacion_ambiental` desde `config.py` (ADR-006).

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_sistemas_informacion_ambiental.html",
       title="EDA — Sistemas de Información Ambiental", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

### ¿Qué esperar en la serie de cobertura de estaciones?

En general, la expansión de redes de monitoreo sigue una **curva logística (S-curve)**:

1. **Fase inicial lenta (pre-2000):** pocas estaciones, concentradas en cuencas prioritarias y ciudades principales.
2. **Fase de crecimiento acelerado (2000–2015):** inversión del IDEAM y CARs, proyectos BID/CAF, digitalización.
3. **Fase de maduración (2015–presente):** crecimiento desacelera porque los sitios más accesibles ya están cubiertos; expandir a Amazonía/Orinoquía es costoso.

**Interpretación visual esperada:**
- Tendencia ascendente general (Mann-Kendall positivo).
- Sin estacionalidad en datos anuales (si aparece ciclicidad en datos mensuales, revisar si es artefacto de reporte).
- Posibles escalones por incorporación masiva de estaciones de una CAR específica.

> La ausencia de ciclo estacional en series anuales justifica usar **ARIMA(1,1,0)** en lugar de SARIMA — ver Sección 7.

In [ ]:
plot_series(df, "fecha", "n_registros", title="Sistemas de Información Ambiental — n_registros ()")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "n_registros", period="month")
plt.show()

## 3c. RETC — Cargas de vertimientos y metales pesados en biosólidos

El **RETC (Registro de Emisiones y Transferencia de Contaminantes)**, integrado al RUA (Res. 839/2023), cuantifica cargas contaminantes vertidas al agua por sector productivo:

| Parametro | Referencia | Umbral reporte |
|---|---|---|
| DBO (vertimientos) | Res. 631/2015 | Segun tipo de actividad |
| SST (vertimientos) | Res. 631/2015 | Segun tipo de actividad |
| Cadmio en biosolidos | Decreto 1287/2014 | < 39 mg/kg MS |
| Plomo en biosolidos | Decreto 1287/2014 | < 300 mg/kg MS |

Colombia vertio ~1.39 M ton DBO y ~1.35 M ton SST en 2020 (IDEAM/SIRH).

> **Tasa retributiva (Decreto 2667/2012):** cargo economico por kg DBO o SST vertido, incentivo economico para que los generadores reduzcan sus cargas.

In [ ]:
# RETC sintetico: cargas de vertimientos por sector + metales en biosolidos
np.random.seed(19)
n_anos = 10
anos = list(range(2014, 2014 + n_anos))
sectores_vert = ['Municipal', 'Agroindustria', 'Mineria', 'Manufactura', 'Servicios']

# Cargas DBO por sector (miles ton DBO/ano)
dbo_kt = {
    'Municipal':     [800 - i*5  + np.random.normal(0, 20) for i in range(n_anos)],
    'Agroindustria': [300 + i*2  + np.random.normal(0, 10) for i in range(n_anos)],
    'Mineria':       [120 - i*1  + np.random.normal(0, 5)  for i in range(n_anos)],
    'Manufactura':   [180 - i*3  + np.random.normal(0, 8)  for i in range(n_anos)],
    'Servicios':     [40  + i*0.5+ np.random.normal(0, 2)  for i in range(n_anos)],
}
df_retc = pd.DataFrame({'ano': anos,
                         **{s: [round(v,1) for v in vals]
                            for s, vals in dbo_kt.items()}})
df_retc['total_dbo_kt'] = df_retc[sectores_vert].sum(axis=1).round(1)

# Metales pesados en biosolidos PTAR (mg/kg MS) -- Decreto 1287/2014
n_ptar = 15
ptars = [f'PTAR-{i+1:02d}' for i in range(n_ptar)]
cadmio = np.random.uniform(10, 95, n_ptar).round(1)   # umbral: 39 mg/kg
plomo  = np.random.uniform(80, 950, n_ptar).round(1)  # umbral: 300 mg/kg
df_biosolidos = pd.DataFrame({'PTAR': ptars, 'Cadmio_mgkg': cadmio, 'Plomo_mgkg': plomo})
df_biosolidos['Cd_cumple'] = df_biosolidos['Cadmio_mgkg'] <= 39
df_biosolidos['Pb_cumple'] = df_biosolidos['Plomo_mgkg'] <= 300

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel A: Cargas DBO vertimientos por sector (stacked)
ax = axes[0]
bottom = np.zeros(n_anos)
colors_vert = ['#e74c3c','#e67e22','#f1c40f','#3498db','#9b59b6']
for s, color in zip(sectores_vert, colors_vert):
    ax.bar(df_retc['ano'], df_retc[s], bottom=bottom, label=s, color=color, alpha=0.85, width=0.7)
    bottom += df_retc[s].values
ax.set_title('RETC — Carga DBO vertida al agua (kt/ano)', fontweight='bold')
ax.set_ylabel('kt DBO/ano'); ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

# Panel B: Cadmio en biosolidos
ax = axes[1]
bar_cd = ['#27ae60' if v <= 39 else '#e74c3c' for v in cadmio]
ax.barh(ptars, cadmio, color=bar_cd, alpha=0.85)
ax.axvline(39, color='red', ls='--', lw=1.5, label='Umbral Cd: 39 mg/kg (Dto. 1287/2014)')
ax.set_title('Cadmio en Biosolidos PTAR', fontweight='bold')
ax.set_xlabel('Cadmio (mg/kg MS)'); ax.legend(fontsize=7); ax.grid(axis='x', alpha=0.3)

# Panel C: Plomo en biosolidos
ax = axes[2]
bar_pb = ['#27ae60' if v <= 300 else '#e74c3c' for v in plomo]
ax.barh(ptars, plomo, color=bar_pb, alpha=0.85)
ax.axvline(300, color='red', ls='--', lw=1.5, label='Umbral Pb: 300 mg/kg (Dto. 1287/2014)')
ax.set_title('Plomo en Biosolidos PTAR', fontweight='bold')
ax.set_xlabel('Plomo (mg/kg MS)'); ax.legend(fontsize=7); ax.grid(axis='x', alpha=0.3)

plt.suptitle('SIAC / RETC — Monitoreo de cargas contaminantes y biosolidos (Res. 839/2023)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

ptar_cd_incumple = (~df_biosolidos['Cd_cumple']).sum()
ptar_pb_incumple = (~df_biosolidos['Pb_cumple']).sum()
print(f'PTAR incumple Cadmio (>39 mg/kg): {ptar_cd_incumple}/{n_ptar}')
print(f'PTAR incumple Plomo (>300 mg/kg): {ptar_pb_incumple}/{n_ptar}')
print(f'Total vertimientos 2023 (estimado): {df_retc["total_dbo_kt"].iloc[-1]:.0f} kt DBO')

## 4. Estadística descriptiva

### ¿Qué buscar en los estadísticos de cobertura?

| Estadístico | Interpretación en este contexto |
|---|---|
| **Media** | Cobertura promedio histórica — ¿supera el 50% del territorio? |
| **Mediana** | Más robusta ante años atípicos de reporte |
| **Skewness** | Si es negativo y alto: la distribución está acumulada en coberturas altas (buena señal). Si es positivo: mayoría de años con baja cobertura |
| **CV (%)** | Variabilidad relativa — un CV >30% sugiere alta heterogeneidad temporal, posiblemente por cambios metodológicos |
| **Mín / Máx** | Identificar años de inicio de serie y posibles errores de digitación |

**Brechas conocidas del SIAC:**
- La zona andina concentra ~70% de las estaciones; la Orinoquía y Amazonía están sistemáticamente subrepresentadas.
- Los estadísticos nacionales ocultan esta heterogeneidad regional — cuando tenga datos desagregados por CAR, calcule descriptivos por región.

In [ ]:
summarize(df)

## 5. Análisis inferencial

### Estacionariedad y tendencia en cobertura de estaciones

**ADR-004:** Se aplican ADF + KPSS de forma obligatoria antes de cualquier modelado ARIMA.

**Interpretación esperada para esta variable:**
- La cobertura de estaciones es una variable acumulativa y con tendencia creciente — es **no estacionaria en nivel** (I(1)).
- ADF: probablemente no rechaza hipótesis nula (raíz unitaria presente).
- KPSS: probablemente rechaza la hipótesis nula (la serie no es estacionaria alrededor de una constante).
- Conclusión: aplicar primera diferencia (d=1) antes de modelar con ARIMA.

**Mann-Kendall sobre la serie original:**
- Hipótesis: H0 = no hay tendencia monótona.
- Resultado esperado: tendencia positiva significativa (p < 0.05), con slope de Theil-Sen positivo.
- Interpretación: la red de monitoreo colombiana ha crecido de forma estadísticamente sostenida en el período analizado.
- Si el slope desacelera en los últimos años: señal de que la curva logística está entrando en la fase de maduración.

> ADR-004: ADF + KPSS obligatorios antes de ARIMA. Ver `inference/stationarity.py`.

In [ ]:
ts = df.set_index("fecha")["n_registros"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas

### ¿Aplica el análisis de excedencias a esta línea?

Para la cobertura de estaciones de monitoreo, no existen umbrales normativos formales en la Resolución 2254/2017 ni en normas de agua (esas normas aplican a parámetros de contaminación). Sin embargo, **sí existen metas institucionales** que pueden tratarse como umbrales:

| Meta institucional | Umbral | Fuente |
|---|---|---|
| Cobertura mínima meta PAI | ≥ 60% del territorio | Plan de Acción Institucional de la CAR |
| Densidad mínima recomendada (OMM) | ≥ 1 est. / 1.000 km² en regiones montañosas | OMM Guía No. 8 |
| Tiempo al dato máximo aceptable | ≤ 30 días | Meta de modernización IDEAM |

Si `exceedance_report()` no retorna resultados para `n_registros` (es esperado — no es una variable de contaminación), use umbrales manuales de referencia institucional.

> Para variables con umbrales propios (pm25, od, etc.), ver `config.NORMA_CO` y `config.NORMA_AGUA_POTABLE`.

In [ ]:
rep = exceedance_report(df["n_registros"], variable="n_registros")
if rep.empty:
    print("Sin normas colombianas registradas para 'n_registros'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

### Manejo de faltantes en series de cobertura institucional

**ADR-002 (Outliers):** No hacer clipping automático. Si la cobertura cae abruptamente un año, puede ser un recorte presupuestal real o un cambio de metodología — ambos son señal valiosa, no ruido.

**Estrategia de imputación recomendada:**
- `method="linear"`: adecuada para series con tendencia suave como la cobertura de estaciones.
- Alternativa: `method="spline"` si hay curvaturas evidentes en la fase de crecimiento.
- **No usar** `method="seasonal"` — la serie es anual y no tiene ciclo estacional.

**Identificación de outliers contextuales:**
- Año con cobertura >5% superior al año anterior: investigar si incorporaron una red de CARs nueva.
- Año con cobertura <5% inferior al año anterior: investigar si hubo desactivación de estaciones o cambio de umbrales de "estación activa".

In [ ]:
df_clean = impute(df.copy(), cols=["n_registros"], method="linear")
print(f"Faltantes antes: {df["n_registros"].isna().sum()} | "
      f"después: {df_clean["n_registros"].isna().sum()}")

## 7. Modelos predictivos — Proyección 5 años

### ¿Por qué ARIMA(1,1,0) y no SARIMA?

**Razón fundamental:** La serie de cobertura de estaciones es **anual**. SARIMA requiere al menos un ciclo estacional completo con periodicidad bien definida (mensual o trimestral). Con datos anuales:
- No existe componente estacional estadísticamente identificable.
- El período estacional mínimo para SARIMA sería 2 observaciones/ciclo, lo que no tiene interpretación física.
- Usar SARIMA en datos anuales añade parámetros sin beneficio — viola el principio de parsimonia.

**Decisión metodológica:** ARIMA(1,1,0) — autoregresivo de orden 1 con una diferenciación.
- `d=1`: porque la serie es I(1) (no estacionaria, con tendencia).
- `p=1`: captura la inercia de año a año (el crecimiento del año pasado predice el crecimiento de este año).
- `q=0`: sin componente de media móvil — el proceso de expansión de redes no tiene correcciones de error de corto plazo.

**Curva logística como alternativa:**  
Si la serie es claramente sigmoidal (crecimiento que desacelera al acercarse a una cobertura máxima), ajustar un modelo logístico:
```
L(t) = K / (1 + exp(-r*(t - t0)))
```
donde K = cobertura máxima plausible (ej. 85%), r = tasa de crecimiento, t0 = punto de inflexión.

**Modelos de ML (RandomForest / XGBoost):**  
Con series anuales cortas (<30 observaciones), los modelos de ML tienen **alta varianza** y poca capacidad de generalización. Son útiles si se tienen covariables relevantes (presupuesto IDEAM anual, número de CARs, km de vías nuevas). Para series cortas sin covariables, ARIMA(1,1,0) es más confiable.

> **Horizonte recomendado:** 5 años (n_splits=3 en walk-forward con series anuales ~20 obs).

In [ ]:
ts = df_clean.set_index("fecha")["n_registros"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

### Plantilla de hallazgos (completar con datos reales)

| Aspecto | Hallazgo | Implicación |
|---|---|---|
| Tendencia | Mann-Kendall: ___ (p=___) | La cobertura de estaciones ___ de forma sostenida |
| Estacionariedad | ADF p=___, KPSS p=___ | Serie I(___), se requiere ___ diferencia(s) para ARIMA |
| Modelo seleccionado | ARIMA(___,___,___) | Proyección para 5 años: ___% en 2030 |
| Cobertura actual | ___% del territorio | ___ con respecto a meta institucional del PAI |
| Brecha regional | ___ estaciones en Amazonía vs. ___ en zona andina | Priorizar inversión en ___ |

### Normativa y decisiones metodológicas registradas

- **ADR-002:** Outliers de cobertura preservados como señal institucional real.
- **ADR-004:** ADF + KPSS aplicados — serie I(1), d=1 en ARIMA.
- **ADR-006:** `validate()` con rangos físicos de cobertura (0–100%).
- **No se usó SARIMA:** serie anual sin componente estacional (ver Sección 7).
- **Documentación de dominio:** `docs/fuentes/sistemas_informacion_ambiental.md`.
- Registrar decisiones adicionales en `docs/decisiones.md`.

---

## 9. Cómo adaptar a datos reales

### Paso 1: Obtener datos del SIAC/IDEAM

```python
# Opción A: descarga manual de portal IDEAM
# http://dhime.ideam.gov.co → Estaciones → Exportar inventario activo

# Opción B: API SIAC (si disponible)
import requests
r = requests.get("http://www.siac.gov.co/api/estaciones", params={"estado": "activa"})
estaciones = r.json()
```

### Paso 2: Calcular cobertura anual

```python
import geopandas as gpd
from shapely.geometry import Point

# Cargar límite nacional (shapefile Colombia)
# Fuente: https://www.datos.gov.co → buscar "Municipios Colombia"
colombia = gpd.read_file("data/raw/Colombia_municipios.shp").dissolve()

# Crear GeoDataFrame de estaciones
gdf = gpd.GeoDataFrame(
    estaciones_df,
    geometry=gpd.points_from_xy(estaciones_df.lon, estaciones_df.lat),
    crs="EPSG:4686"   # MAGNA-SIRGAS
)

# Buffer de 50 km por estación → unión → intersección con Colombia
buffer = gdf.to_crs("EPSG:9377").buffer(50_000)  # metros en CTM12
cobertura_area = buffer.union_all()
cobertura_pct = cobertura_area.intersection(colombia.to_crs("EPSG:9377").geometry.iloc[0]).area / \
                colombia.to_crs("EPSG:9377").area.iloc[0] * 100
print(f"Cobertura: {cobertura_pct:.1f}%")
```

### Paso 3: Agregar por año y cargar en este notebook

```python
# Reemplazar datos sintéticos:
df = pd.read_csv("data/raw/cobertura_estaciones_anual.csv", parse_dates=["fecha"])
# Columnas esperadas: fecha (YYYY-01-01), cobertura_pct, densidad_est_km2
```

### Fuentes de datos adicionales

- **SIRH/DHIME:** inventario histórico de estaciones hidrometeorológicas.
- **RUNAP:** registro de estaciones en áreas protegidas.
- **CARs:** solicitar consolidado de estaciones propias vía derecho de petición.
- **OMM (Guía No. 8):** densidades mínimas de estaciones por tipo de región.